In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import(
    col,
    row_number,
    trim
)
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

#### Importing validation class from the test_utils module

In [0]:
from validation_utils.test_utils import Validations
obj = Validations()

#### Defining orchestrator function that runs all the test functions inside the test_utils

In [0]:
def orchestrate_tests(
    table: str,
    obj: Validations,
    primary_col: str,
    dedup_cols: list,
    cdc: str
) -> None:
    df = spark.table(f"lakehouse.silver.{table}")

    def log(msg: str) -> None:
        print(f"\n{'-' * 20}\t{msg}\t{'-' * 20}\n")

    log("Null Check Start")
    obj.null_check(df, primary_col)
    log("Null Check End")

    log("Duplicate Check Start")
    obj.duplicate_check(df, dedup_cols, cdc)
    log("Duplicate Check End")

    log("Whitespace Check Start")
    obj.whitespace_check(df)
    log("Whitespace Check End")



##### 1. crm_customers

In [0]:
orchestrate_tests(
    table = "crm_customers",
    obj = obj,
    primary_col = 'customer_id',
    dedup_cols = ['customer_id'],
    cdc = 'create_date'
)

#### 2. crm_products

In [0]:
orchestrate_tests(
    table = "crm_products",
    obj = obj,
    primary_col = 'product_id',
    dedup_cols = ['product_id'],
    cdc = 'start_date'
)

##### Checking if there are nulls in product_cost

In [0]:
%sql
SELECT *
FROM lakehouse.silver.crm_products
WHERE product_cost IS NULL

##### Checking if end_date < start_date

In [0]:
%sql
SELECT *
FROM lakehouse.silver.crm_products
WHERE end_date < start_date